# 📱 ISOM 260: Your Agent, On Your Phone

On Wednesday you wrote the agent loop. It lived in a Colab cell and answered when you ran the cell.

**Today it moves out.** In about 15 minutes you will have a Telegram bot on your phone that is *your* agent: your tools, your Gemini key, your rules. Text it from the bus. Hand your phone to a friend and let them try to break it.

**How this works — read this once:**

| Piece | Where it runs | What it does |
|---|---|---|
| Telegram app | your phone | the *interface* — just a chat window |
| Telegram's servers | the internet | hold your messages until your bot picks them up |
| this notebook | Colab (for now) | the *agent* — the loop, the tools, the memory |
| Gemini | Google | the *brain* |

Your bot is alive **while the last cell of this notebook is running**. Close the tab and it goes to sleep. At the end you download a `bot.py` that runs on your laptop — or on a $5 server — with no Colab at all. Same code.

> 🔐 **Privacy, in one line:** everything you or your friends text the bot passes through Telegram and Google. Fictional résumés only. Nothing is saved to disk in this notebook.

## 🚀 Part 1: Setup (2 minutes)

You need **two secrets** in the 🔑 panel on the left, both with *Notebook access* switched on:

1. `GOOGLE_API_KEY` — you already have this from Session 3.
2. `TELEGRAM_BOT_TOKEN` — new. Get it in 60 seconds:
   - On your phone, open Telegram and search for **@BotFather** (blue check mark).
   - Send `/newbot`. Give it a display name (e.g. *Hasan's Agent*) and a username that ends in `bot` (e.g. `hasan_isom260_bot`).
   - BotFather replies with a token that looks like `8123456789:AAF...`. **That token is a password.** Copy it into Colab Secrets — never into a cell, never into a screenshot.
3. *(Optional, 3 minutes)* `ADZUNA_APP_ID` and `ADZUNA_APP_KEY` — free at [developer.adzuna.com](https://developer.adzuna.com). This unlocks **local** listings ("analyst jobs in Boston"). Without it the bot still works; local searches fall back to remote listings and say so.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata

try:
    GOOGLE_API_KEY     = userdata.get("GOOGLE_API_KEY")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    print("✅ Both secrets loaded from Colab Secrets.")
except Exception:
    raise SystemExit("❌ Missing a secret. Open the 🔑 panel on the left, add GOOGLE_API_KEY and "
                     "TELEGRAM_BOT_TOKEN, switch on 'Notebook access' for both, then re-run this cell.")

# Optional third secret pair: Adzuna, for LOCAL job listings (see Part 1, step 3)
try:
    ADZUNA_APP_ID, ADZUNA_APP_KEY = userdata.get("ADZUNA_APP_ID"), userdata.get("ADZUNA_APP_KEY")
    print("✅ Adzuna key found — local job search is live.")
except Exception:
    ADZUNA_APP_ID = ADZUNA_APP_KEY = None
    print("ℹ️  No Adzuna key — local job search will fall back to remote listings (and say so).")

In [ ]:
from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"

def tg(method, http_timeout=20, **params):
    """One function for the whole Telegram Bot API: tg('sendMessage', chat_id=..., text=...)"""
    req = urllib.request.Request(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/{method}",
                                 data=json.dumps(params).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=http_timeout))["result"]

me = tg("getMe")
BOT_LINK = f"https://t.me/{me['username']}"
print(f"🧠 Gemini ready · 🤖 bot is alive: @{me['username']}\n   Open on your phone → {BOT_LINK}")

## 🛠️ Part 2: Tools (the part you already know)

Same shape as Session 4: a **job description** Gemini reads, and a **Python function** your code runs.

Three tools are pre-loaded. Two are real: one hits a public remote-job board with no key, one searches local postings through Adzuna **if** you added the optional key — and if not, it falls back to the remote board and *labels the result as a fallback* so the model cannot pass remote jobs off as local. That label is the whole trick behind "provider abstraction": the agent's workflow never changes, only where the data comes from.

**Paste your HW #5 tool** into the marked slot and register it — that is what makes this bot *yours*.

In [ ]:
# ============================================
# TOOLS — the same shape as Session 4: a job description + a Python function
# ============================================

# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: LOCAL jobs via Adzuna (optional key) — falls back to the remote board, and says so
adzuna_tool = types.FunctionDeclaration(
    name="find_local_jobs",
    description=(
        "Search real job postings near a US city (default Boston) by keyword, e.g. 'business analyst'. "
        "Returns up to 5 openings with company, location, salary and a link, plus a 'source' field. "
        "If source says FALLBACK, the listings are REMOTE jobs, not local — tell the user that plainly."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword":  {"type": "string", "description": "Job title or skill, e.g. 'financial analyst'"},
                       "location": {"type": "string", "description": "City, e.g. 'Boston'. Default Boston."}},
        "required": ["keyword"],
    },
)

def find_local_jobs(keyword: str, location: str = "Boston", limit: int = 5) -> str:
    def fallback(why):
        return json.dumps({"source": f"FALLBACK ({why}) — these are REMOTE listings, not jobs in {location}",
                           "jobs": json.loads(find_jobs(keyword, limit))})
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return fallback("no Adzuna key")
    try:
        q = urllib.parse.urlencode({"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY, "what": keyword,
                                    "where": location, "results_per_page": limit, "content-type": "application/json"})
        req = urllib.request.Request(f"https://api.adzuna.com/v1/api/jobs/us/search/1?{q}",
                                     headers={"User-Agent": "ISOM260-student-project"})
        raw = json.load(urllib.request.urlopen(req, timeout=20))
        jobs = []
        for r in raw.get("results", [])[:limit]:
            sal = f"${int(r['salary_min']):,}-${int(r.get('salary_max') or r['salary_min']):,}" if r.get("salary_min") else "not listed"
            jobs.append({"title": r.get("title", "?"), "company": (r.get("company") or {}).get("display_name", "?"),
                         "location": (r.get("location") or {}).get("display_name", location), "salary": sal,
                         "url": r.get("redirect_url", ""), "snippet": " ".join((r.get("description") or "").split())[:200]})
        return json.dumps({"source": f"Adzuna live · {raw.get('count', len(jobs))} postings match near {location}", "jobs": jobs})
    except Exception as e:
        return fallback(f"Adzuna failed: {type(e).__name__}")


# Tool 4: YOUR HW #5 tool goes here (paste the declaration + function, then register it below)


TOOLS     = [calculator_tool, jobs_tool, adzuna_tool]                     # ← add your tool's declaration
FUNCTIONS = {"calculator": calculator, "find_jobs": find_jobs, "find_local_jobs": find_local_jobs}   # ← and its function

print(f"🛠️  {len(TOOLS)} tools registered: {', '.join(FUNCTIONS)}")
print("   calculator('7394 * 8261') →", calculator("7394 * 8261"))
print("   find_local_jobs('analyst') source →", json.loads(find_local_jobs("analyst", limit=1))["source"])

## 🧠 Part 3: The agent — with memory

This is Wednesday's loop with **one addition**: a `MEMORY` dictionary keyed by `chat_id`. Every phone that talks to your bot gets its own conversation history, so your friend's questions never leak into yours.

Look for the three trust-layer pieces you already met: retries, a circuit breaker (`MAX_STEPS`), and an audit log that records *which tool was called*, never *what was said*.

In [ ]:
# ============================================
# THE AGENT — Session 4's loop, plus one new thing: MEMORY per chat
# ============================================
SYSTEM = (
    "You are a personal assistant living inside a Telegram chat on the user's phone. "
    "Keep answers short and skimmable — it is a phone screen. Use tools whenever they improve accuracy. "
    "If a tool returns an error, say so honestly and never invent a result. "
    "If a job tool reports a FALLBACK source, say plainly that the listings are remote, not local."
)
MEMORY    = {}      # chat_id → conversation so far. Lives only while the bot cell runs — nothing is saved to disk.
AUDIT_LOG = []      # every tool call (never the message text itself)
MAX_STEPS = 8       # circuit breaker
MAX_TURNS = 12      # remember the last 12 user messages per chat, then forget the oldest


def call_model_with_retry(contents, config, max_retries=4):
    """Free tiers rate-limit. Wait and retry instead of crashing."""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt)
                print(f"   ⏳ API returned {e.code} — waiting {wait}s")
                time.sleep(wait)
            else:
                raise


def trim(history):
    """Keep the last MAX_TURNS user messages (and everything after them) so the context window stays small."""
    starts = [i for i, c in enumerate(history)
              if c.role == "user" and c.parts and c.parts[0].text is not None]
    if len(starts) > MAX_TURNS:
        del history[:starts[-MAX_TURNS]]


def agent_reply(chat_id, text):
    history = MEMORY.setdefault(chat_id, [])
    history.append(types.Content(role="user", parts=[types.Part(text=text)]))
    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=TOOLS)], system_instruction=SYSTEM)

    for step in range(1, MAX_STEPS + 1):
        response = call_model_with_retry(history, config)
        model_content = response.candidates[0].content
        history.append(model_content)
        calls = [p for p in (model_content.parts or []) if p.function_call]

        if not calls:                                   # final answer → back to the phone
            trim(history)
            return response.text or "(I came back empty — try rephrasing?)"

        result_parts = []
        for part in calls:
            name, args = part.function_call.name, dict(part.function_call.args)
            t0 = time.time()
            try:
                result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
            except Exception as e:
                result = f"Error: tool '{name}' failed: {e}"
            AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"),
                              "chat": str(chat_id)[-4:], "step": step, "tool": name,
                              "args": json.dumps(args, default=str)[:60],
                              "result": str(result)[:60], "ms": round((time.time() - t0) * 1000)})
            result_parts.append(types.Part.from_function_response(name=name, response={"result": result}))
        history.append(types.Content(role="user", parts=result_parts))

    return "⚠️ I stopped after too many steps without reaching an answer. Try asking it differently?"

# Quick test — same as Session 4, no phone yet
print(agent_reply("test", "What is 7,394 multiplied by 8,261?"))
print(agent_reply("test", "And divided by 2?"))       # ← memory: 'it' refers to the previous answer
MEMORY.clear()

## 📡 Part 4: The bridge — put it on your phone

Telegram keeps your messages in a queue. The bot asks *"anything new?"*, waits up to 30 seconds for an answer, hands each message to `agent_reply`, sends the reply back, and asks again. That is the entire app.

**Run the cell, then open your bot on your phone and say hi.** The cell keeps running — that *is* the bot. Press the ■ stop button to shut it down.

In [ ]:
# ============================================
# THE BRIDGE — Telegram in, agent, Telegram out. ~40 lines. This is the whole "app".
# ============================================
HELP = ("👋 I'm your agent. I can do math and search real jobs — remote, or near a city. Ask me anything.\n"
        "/reset makes me forget our conversation.")

def send(chat_id, text):
    for i in range(0, len(text), 4000):                 # Telegram caps one message at 4,096 characters
        tg("sendMessage", chat_id=chat_id, text=text[i:i + 4000])

def handle(msg):
    chat_id, text = msg["chat"]["id"], msg.get("text")
    if not text:
        send(chat_id, "I only read text for now 🙂"); return
    if text.startswith(("/start", "/help")):
        send(chat_id, HELP); return
    if text.startswith("/reset"):
        MEMORY.pop(chat_id, None); send(chat_id, "🧹 Fresh start. What's up?"); return
    tg("sendChatAction", chat_id=chat_id, action="typing")
    try:
        reply = agent_reply(chat_id, text)
    except Exception as e:                              # the bot must never die because one answer failed
        reply = f"😵 Something went wrong on my side ({type(e).__name__}). Try again?"
    send(chat_id, reply)

def run_bot():
    print(f"📱 Listening. Open {BOT_LINK} on your phone and say hi.  (Press ■ to stop.)")
    offset, served = 0, 0
    try:
        while True:                                     # observe → decide → act → observe, forever
            try:
                updates = tg("getUpdates", http_timeout=45, offset=offset, timeout=30, allowed_updates=["message"])
            except Exception as e:                      # Wi-Fi hiccup — wait, then keep listening
                print(f"   ⏳ polling hiccup ({type(e).__name__}) — retrying in 5s"); time.sleep(5); continue
            for u in updates:
                offset = u["update_id"] + 1
                if "message" in u:
                    served += 1
                    who = u["message"]["chat"].get("first_name", "someone")
                    print(f"{datetime.datetime.now():%H:%M:%S}  💬 #{served} from {who}")   # who, never what
                    handle(u["message"])
    except KeyboardInterrupt:
        print(f"\n🛑 Bot stopped after {served} message(s). Run this cell again to bring it back.")

run_bot()

## 🧪 Part 5: Try to break it (5 minutes)

While your bot is running, from your phone:

1. `What's 15% of $4,500 plus $2,800?` — it should reach for the calculator.
2. `Find me marketing jobs` — a real tool, real listings, right now.
3. `Which of those pays the most?` — memory: it should answer *without* searching again.
4. `Entry-level analyst jobs in Boston` — with an Adzuna key: local postings. Without: remote ones, and the bot must *say* they are remote. Did it?
5. **Swap phones with a neighbor.** Their bot, your question. Two phones, two separate memories.
6. Ask it something your HW #5 tool handles. Did it choose the right tool?

Then stop the bot (■) and look at the receipt:

In [ ]:
import pandas as pd
pd.DataFrame(AUDIT_LOG) if AUDIT_LOG else print("No tool calls yet — run the bot and text it first.")

## 🚚 Part 6: Actually outside Colab

Colab was the server today. It sleeps when you close the tab. The cell below writes **`bot.py`** — the exact code from Parts 2–4, one file — and downloads it.

On your laptop (Python 3.10+):

```
pip install google-genai
export GOOGLE_API_KEY=...  TELEGRAM_BOT_TOKEN=...      # Windows: set GOOGLE_API_KEY=...
python bot.py
```

Your bot is now up as long as your laptop is. Want it up 24/7? The same file runs unchanged on any small cloud box — a free-tier worker on Railway or Render, or a $5/month VM. That is a 20-minute office-hours job, not a course unit.

> ⚠️ **Only one copy at a time.** If `bot.py` and this notebook both poll the same token, Telegram gives each of them half the messages. Stop one before you start the other.

In [ ]:
BOT_PY = r'''"""bot.py — your Telegram agent, standalone. Runs anywhere Python runs.

    pip install google-genai
    export GOOGLE_API_KEY=...  TELEGRAM_BOT_TOKEN=...     (Windows: set GOOGLE_API_KEY=...)
    export ADZUNA_APP_ID=...   ADZUNA_APP_KEY=...          (optional: local job search)
    python bot.py
"""
import os
from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

GOOGLE_API_KEY     = os.environ["GOOGLE_API_KEY"]
TELEGRAM_BOT_TOKEN = os.environ["TELEGRAM_BOT_TOKEN"]
ADZUNA_APP_ID      = os.environ.get("ADZUNA_APP_ID")       # optional — local job search
ADZUNA_APP_KEY     = os.environ.get("ADZUNA_APP_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"

def tg(method, http_timeout=20, **params):
    """One function for the whole Telegram Bot API: tg('sendMessage', chat_id=..., text=...)"""
    req = urllib.request.Request(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/{method}",
                                 data=json.dumps(params).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=http_timeout))["result"]

BOT_LINK = "https://t.me/" + tg("getMe")["username"]

# ============================================
# TOOLS — the same shape as Session 4: a job description + a Python function
# ============================================

# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: LOCAL jobs via Adzuna (optional key) — falls back to the remote board, and says so
adzuna_tool = types.FunctionDeclaration(
    name="find_local_jobs",
    description=(
        "Search real job postings near a US city (default Boston) by keyword, e.g. 'business analyst'. "
        "Returns up to 5 openings with company, location, salary and a link, plus a 'source' field. "
        "If source says FALLBACK, the listings are REMOTE jobs, not local — tell the user that plainly."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword":  {"type": "string", "description": "Job title or skill, e.g. 'financial analyst'"},
                       "location": {"type": "string", "description": "City, e.g. 'Boston'. Default Boston."}},
        "required": ["keyword"],
    },
)

def find_local_jobs(keyword: str, location: str = "Boston", limit: int = 5) -> str:
    def fallback(why):
        return json.dumps({"source": f"FALLBACK ({why}) — these are REMOTE listings, not jobs in {location}",
                           "jobs": json.loads(find_jobs(keyword, limit))})
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return fallback("no Adzuna key")
    try:
        q = urllib.parse.urlencode({"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY, "what": keyword,
                                    "where": location, "results_per_page": limit, "content-type": "application/json"})
        req = urllib.request.Request(f"https://api.adzuna.com/v1/api/jobs/us/search/1?{q}",
                                     headers={"User-Agent": "ISOM260-student-project"})
        raw = json.load(urllib.request.urlopen(req, timeout=20))
        jobs = []
        for r in raw.get("results", [])[:limit]:
            sal = f"${int(r['salary_min']):,}-${int(r.get('salary_max') or r['salary_min']):,}" if r.get("salary_min") else "not listed"
            jobs.append({"title": r.get("title", "?"), "company": (r.get("company") or {}).get("display_name", "?"),
                         "location": (r.get("location") or {}).get("display_name", location), "salary": sal,
                         "url": r.get("redirect_url", ""), "snippet": " ".join((r.get("description") or "").split())[:200]})
        return json.dumps({"source": f"Adzuna live · {raw.get('count', len(jobs))} postings match near {location}", "jobs": jobs})
    except Exception as e:
        return fallback(f"Adzuna failed: {type(e).__name__}")


# Tool 4: YOUR HW #5 tool goes here (paste the declaration + function, then register it below)


TOOLS     = [calculator_tool, jobs_tool, adzuna_tool]                     # ← add your tool's declaration
FUNCTIONS = {"calculator": calculator, "find_jobs": find_jobs, "find_local_jobs": find_local_jobs}   # ← and its function

# ============================================
# THE AGENT — Session 4's loop, plus one new thing: MEMORY per chat
# ============================================
SYSTEM = (
    "You are a personal assistant living inside a Telegram chat on the user's phone. "
    "Keep answers short and skimmable — it is a phone screen. Use tools whenever they improve accuracy. "
    "If a tool returns an error, say so honestly and never invent a result. "
    "If a job tool reports a FALLBACK source, say plainly that the listings are remote, not local."
)
MEMORY    = {}      # chat_id → conversation so far. Lives only while the bot cell runs — nothing is saved to disk.
AUDIT_LOG = []      # every tool call (never the message text itself)
MAX_STEPS = 8       # circuit breaker
MAX_TURNS = 12      # remember the last 12 user messages per chat, then forget the oldest


def call_model_with_retry(contents, config, max_retries=4):
    """Free tiers rate-limit. Wait and retry instead of crashing."""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt)
                print(f"   ⏳ API returned {e.code} — waiting {wait}s")
                time.sleep(wait)
            else:
                raise


def trim(history):
    """Keep the last MAX_TURNS user messages (and everything after them) so the context window stays small."""
    starts = [i for i, c in enumerate(history)
              if c.role == "user" and c.parts and c.parts[0].text is not None]
    if len(starts) > MAX_TURNS:
        del history[:starts[-MAX_TURNS]]


def agent_reply(chat_id, text):
    history = MEMORY.setdefault(chat_id, [])
    history.append(types.Content(role="user", parts=[types.Part(text=text)]))
    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=TOOLS)], system_instruction=SYSTEM)

    for step in range(1, MAX_STEPS + 1):
        response = call_model_with_retry(history, config)
        model_content = response.candidates[0].content
        history.append(model_content)
        calls = [p for p in (model_content.parts or []) if p.function_call]

        if not calls:                                   # final answer → back to the phone
            trim(history)
            return response.text or "(I came back empty — try rephrasing?)"

        result_parts = []
        for part in calls:
            name, args = part.function_call.name, dict(part.function_call.args)
            t0 = time.time()
            try:
                result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
            except Exception as e:
                result = f"Error: tool '{name}' failed: {e}"
            AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"),
                              "chat": str(chat_id)[-4:], "step": step, "tool": name,
                              "args": json.dumps(args, default=str)[:60],
                              "result": str(result)[:60], "ms": round((time.time() - t0) * 1000)})
            result_parts.append(types.Part.from_function_response(name=name, response={"result": result}))
        history.append(types.Content(role="user", parts=result_parts))

    return "⚠️ I stopped after too many steps without reaching an answer. Try asking it differently?"

# ============================================
# THE BRIDGE — Telegram in, agent, Telegram out. ~40 lines. This is the whole "app".
# ============================================
HELP = ("👋 I'm your agent. I can do math and search real jobs — remote, or near a city. Ask me anything.\n"
        "/reset makes me forget our conversation.")

def send(chat_id, text):
    for i in range(0, len(text), 4000):                 # Telegram caps one message at 4,096 characters
        tg("sendMessage", chat_id=chat_id, text=text[i:i + 4000])

def handle(msg):
    chat_id, text = msg["chat"]["id"], msg.get("text")
    if not text:
        send(chat_id, "I only read text for now 🙂"); return
    if text.startswith(("/start", "/help")):
        send(chat_id, HELP); return
    if text.startswith("/reset"):
        MEMORY.pop(chat_id, None); send(chat_id, "🧹 Fresh start. What's up?"); return
    tg("sendChatAction", chat_id=chat_id, action="typing")
    try:
        reply = agent_reply(chat_id, text)
    except Exception as e:                              # the bot must never die because one answer failed
        reply = f"😵 Something went wrong on my side ({type(e).__name__}). Try again?"
    send(chat_id, reply)

def run_bot():
    print(f"📱 Listening. Open {BOT_LINK} on your phone and say hi.  (Press ■ to stop.)")
    offset, served = 0, 0
    try:
        while True:                                     # observe → decide → act → observe, forever
            try:
                updates = tg("getUpdates", http_timeout=45, offset=offset, timeout=30, allowed_updates=["message"])
            except Exception as e:                      # Wi-Fi hiccup — wait, then keep listening
                print(f"   ⏳ polling hiccup ({type(e).__name__}) — retrying in 5s"); time.sleep(5); continue
            for u in updates:
                offset = u["update_id"] + 1
                if "message" in u:
                    served += 1
                    who = u["message"]["chat"].get("first_name", "someone")
                    print(f"{datetime.datetime.now():%H:%M:%S}  💬 #{served} from {who}")   # who, never what
                    handle(u["message"])
    except KeyboardInterrupt:
        print(f"\n🛑 Bot stopped after {served} message(s). Run this cell again to bring it back.")

if __name__ == "__main__":
    run_bot()
'''

with open("bot.py", "w") as f:
    f.write(BOT_PY)
from google.colab import files
files.download("bot.py")
print("⬇️  bot.py downloaded —", len(BOT_PY.splitlines()), "lines, zero Colab.")

## 🎓 What you shipped — and what you didn't

**You shipped:** a personal AI agent with tools, per-user memory, retries, a circuit breaker, an audit log, and a real interface strangers can use. In production the *interface* is often the least interesting part — Slack, WhatsApp, a web form, and Telegram all look the same to the loop.

**You did not ship:** persistence (memory dies with the cell — a database fixes that), an approval gate that works over chat (the agent would have to *ask you* on Telegram before a risky action — next session), or authentication (anyone who finds your bot can use your Gemini quota — think about that before you post the link).

**Boardroom sentence:** *"The agent is 40 lines. The interface is 40 lines. The other 90% of the budget is trust: memory, permissions, audit, and what happens when a tool fails."*